# Benchmark M2 trên Google Colab (CPU)

Đề M2 chốt testbed benchmark là Google Colab CPU. Notebook này chạy Parser M2 (`scripts.parse_range`) trên **cả Range của `MEMBER`** và đo:
- thời gian end-to-end và thời gian trung bình mỗi Publication;
- RAM tối đa, đo bằng `/usr/bin/time -v` (cả tiến trình);
- dung lượng đầu ra.

Kết quả được upload lên HF `benchmarks-m2/<MEMBER>/<thời điểm>/`. Một Range mất khoảng 30–60 phút: tải 2 file nén (vài trăm MB) rồi parse.

**Cách chạy**
1. *Runtime → Change runtime type → **CPU***.
2. 🔑 **Secrets**: bật *Notebook access* cho `HF_TOKEN` (đọc và ghi dataset `hcmus-arxiv-reference/arXiv-reference-matching`) và `HF_REPO_ID` = `hcmus-arxiv-reference/arXiv-reference-matching`.
3. Ô **Tham số**: đặt `MEMBER`, rồi *Runtime → Run all*.

Chỉ sửa ô **Tham số**. Không dán token vào ô code.

In [ ]:
# ==== Tham số ====
MEMBER = "bao"          # bao, minh, nghiem, hai, lieu
REF = "main"            # nhánh, tag hoặc SHA của repo code
WORKERS = 1             # Colab CPU có 2 vCPU; song song không nhanh hơn (Book M2, trang 07)
REPO_URL = "https://github.com/hcmus-arxiv-reference/arxiv-reference-matching.git"
WORK = "/content/m2"
UPLOAD_RESULTS = True

In [ ]:
import json, os, platform, shutil, subprocess, sys, tarfile, time
from datetime import datetime, timezone
from pathlib import Path

STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H%M%SZ")
REPO = Path("/content/arxiv-reference-matching")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin", REF], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "--detach", "FETCH_HEAD"], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
os.chdir(REPO); sys.path.insert(0, str(REPO))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("commit:", COMMIT)

In [ ]:
# Bí mật lấy từ Colab Secrets; không in giá trị.
try:
    from google.colab import userdata
    for name in ("HF_TOKEN", "HF_REPO_ID"):
        os.environ[name] = userdata.get(name)
except ImportError:
    pass
assert os.environ.get("HF_TOKEN") and os.environ.get("HF_REPO_ID"), "Thêm HF_TOKEN và HF_REPO_ID trong Secrets"
print("HF_REPO_ID:", os.environ["HF_REPO_ID"])

In [ ]:
# Tải corpus M1 của Range và sidecar bibliography: mỗi thứ một file .tar.gz trên dataset private
# (inputs/corpus/<MEMBER>.tar.gz gồm metadata/ và raw/; inputs/supplement/<MEMBER>.tar.gz).
# Không dùng snapshot_download: hàm đó liệt kê cả repo (hàng trăm nghìn file raw) trước khi lọc.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "60")   # request treo quá 60 giây thì báo lỗi, không đứng mãi
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
from huggingface_hub import hf_hub_download

t0 = time.monotonic()
DL = Path(WORK, "download"); CORPUS = Path(WORK, "corpus"); SUPP = Path(WORK, "supplement")
for target in (CORPUS, SUPP):
    if target.exists():
        shutil.rmtree(target)
for name, target in ((f"inputs/corpus/{MEMBER}.tar.gz", CORPUS), (f"inputs/supplement/{MEMBER}.tar.gz", SUPP)):
    archive = hf_hub_download(os.environ["HF_REPO_ID"], name, repo_type="dataset",
                              token=os.environ["HF_TOKEN"], local_dir=DL)
    with tarfile.open(archive) as tar:
        tar.extractall(target, filter="data")
    # File AppleDouble "._*" do tar trên macOS sinh ra không phải dữ liệu; để lại thì Version không
    # còn khớp receipt source.json và bị coi là chưa xác minh.
    junk = [p for p in target.rglob("._*") if p.is_file()]
    for p in junk:
        p.unlink()
    print("đã giải nén", name, f"(bỏ {len(junk)} file ._*)" if junk else "")
DOWNLOAD_SECONDS = time.monotonic() - t0
print(f"{len(list((CORPUS / 'metadata').iterdir()))} Publication, tải và giải nén trong {DOWNLOAD_SECONDS/60:.1f} phút")

In [ ]:
# Chạy M2 cả Range. /usr/bin/time -v đo RAM tối đa của tiến trình; metrics.json do pipeline ghi.
OUT = Path(WORK, "out", STAMP); OUT.mkdir(parents=True)
cmd = ["/usr/bin/time", "-v", sys.executable, "-m", "scripts.parse_range", "--data-dir", str(CORPUS),
       "--member", MEMBER, "--out-dir", str(OUT / "parse"), "--supplement-dir", str(SUPP / MEMBER),
       "--roots", "configs/m2-reviewed-roots.json", "--metrics", str(OUT / "metrics.json"), "--no-resume"]
if WORKERS > 1:   # --workers có từ PR #63; mặc định 1 chạy được cả trên commit chưa có tùy chọn này
    cmd += ["--workers", str(WORKERS)]
t0 = time.monotonic()
with open(OUT / "run.log", "w") as log, open(OUT / "time.txt", "w") as timing:
    code = subprocess.run(cmd, stdout=log, stderr=timing).returncode
WALL = time.monotonic() - t0
print("exit", code, f"- {WALL/60:.1f} phút")
assert code == 0, (OUT / "run.log").read_text()[-1500:] + (OUT / "time.txt").read_text()[:1500]

In [ ]:
# Tổng hợp số đo theo yêu cầu của đề M2.
import psutil
m = json.loads((OUT / "metrics.json").read_text())
max_rss_kb = int(next(l.split(":")[1] for l in (OUT / "time.txt").read_text().splitlines()
                      if "Maximum resident set size" in l))
summary = {
    "member": MEMBER, "commit": COMMIT, "stamp": STAMP, "testbed": "Google Colab CPU",
    "machine": {"platform": platform.platform(), "cpus": os.cpu_count(),
                "ram_gib": round(psutil.virtual_memory().total / 2**30, 1)},
    "workers": WORKERS,
    "download_seconds": round(DOWNLOAD_SECONDS, 1),
    "parse_wall_seconds": round(WALL, 1),
    "publications": m["publications_total"],
    "avg_seconds_per_publication": round(WALL / m["publications_total"], 3),
    "max_rss_mb": round(max_rss_kb / 1024, 1),
    "tables_mb": m["tables_dir_size_mb"], "output_mb": m["output_dir_size_mb"],
    "publications_succeeded": m["publications_succeeded"], "versions_parsed": m["versions_parsed"],
    "elements_after_dedup": m["elements_after_dedup"], "bibtex_entries_after_dedup": m["bibtex_entries_after_dedup"],
}
(OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1))
print(json.dumps(summary, ensure_ascii=False, indent=1))

In [ ]:
# Upload số đo (không upload bảng: bảng chính thức đã ở tables/<MEMBER>/).
if UPLOAD_RESULTS:
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ["HF_TOKEN"])
    for name in ("summary.json", "metrics.json", "time.txt", "run.log"):
        api.upload_file(path_or_fileobj=str(OUT / name), path_in_repo=f"benchmarks-m2/{MEMBER}/{STAMP}/{name}",
                        repo_id=os.environ["HF_REPO_ID"], repo_type="dataset",
                        commit_message=f"M2 Colab benchmark {MEMBER} {STAMP}")
    print("Đã upload: benchmarks-m2/%s/%s/" % (MEMBER, STAMP))
print("TRẠNG THÁI: XONG")